In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from shapely import wkt
from shapely.geometry import Point

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output, Javascript
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

# ===========================================================================
# 1) LOAD MODEL
# ===========================================================================
try:
    with open('best_xgboost_spatial_model.pkl', 'rb') as file:
        loaded_model = pickle.load(file)
except FileNotFoundError:
    print("Error: File 'best_xgboost_spatial_model.pkl' tidak ditemukan!")

# ===========================================================================
# 2) HELPER FUNCTIONS
# ===========================================================================
def calculate_angle(p1, p2):
    """Menghitung sudut dari p1 ke p2 dalam derajat."""
    return np.degrees(np.arctan2(p2.y - p1.y, p2.x - p1.x))

def parse_latlon(text):
    """Parsing input string 'lat,lon' menjadi float tuple."""
    if isinstance(text, (tuple, list)) and len(text) == 2:
        return float(text[0]), float(text[1])
    if not isinstance(text, str):
        raise ValueError("Input lat/lon harus string format 'lat,lon'.")
    parts = [p.strip() for p in text.split(",")]
    if len(parts) != 2:
        raise ValueError("Format harus 'latitude,longitude'.")
    return float(parts[0]), float(parts[1])

def build_features(df):
    """
    Transformasi data mentah menjadi fitur spasial:
    - Reproyeksi ke UTM 48S (EPSG:32748)
    - Jarak ke centroid, status 'within', dan sudut pergerakan.
    """
    df = df.copy()
    df["geometry"] = df["SHAPE"].apply(wkt.loads)
    gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")
    # gdf['GPS_PRECISION_SCORE'] = (1 - (gdf['AKURASI'] / gdf['AKURASI'].max())) * 100
    gdf['GPS_PRECISION_SCORE'] = np.clip(100 - (gdf['AKURASI'] * 10), 0, 100)


    # Konversi ke meter untuk perhitungan jarak akurat
    gdf_meter = gdf.to_crs(epsg=32748)
    centroids_meter = gdf_meter.geometry.centroid
    p_meters = []

    for i in range(1, 4):
        p_geom = [Point(xy) for xy in zip(gdf[f"Longitude_{i}"], gdf[f"Latitude_{i}"])]
        p_gdf = gpd.GeoDataFrame(geometry=p_geom, crs="EPSG:4326", index=gdf.index).to_crs(epsg=32748)
        p_meters.append(p_gdf.geometry)

        gdf[f"in_{i}"] = p_gdf.within(gdf_meter.geometry).astype(int)
        gdf[f"dist_c{i}"] = p_gdf.distance(centroids_meter)
        gdf[f"angle_{i}"] = [calculate_angle(c, p) for c, p in zip(centroids_meter, p_gdf.geometry)]

    gdf["all_inside"] = ((gdf["in_1"] == 1) & (gdf["in_2"] == 1) & (gdf["in_3"] == 1)).astype(int)
    gdf["move_angle_12"] = [calculate_angle(p1, p2) for p1, p2 in zip(p_meters[0], p_meters[1])]
    gdf["move_angle_23"] = [calculate_angle(p2, p3) for p2, p3 in zip(p_meters[1], p_meters[2])]

    features = [
        "in_1", "in_2", "in_3", "all_inside",
        "dist_c1", "dist_c2", "dist_c3",
        "angle_1", "angle_2", "angle_3",
        "move_angle_12", "move_angle_23",
        "GPS_PRECISION_SCORE", "JUMLAH_SATELIT"
    ]
    return gdf, gdf[features]

def predict_and_plot(input_df):
    """Main function untuk prediksi dan visualisasi dengan threshold 80%."""
    gdf, X_feat = build_features(input_df)
    
    # Ekstrak nilai probabilitas untuk kelas positif (1)
    probability = loaded_model.predict_proba(X_feat)[:, 1]
    
    # Timpa nilai prediction default dengan threshold kustom (0.70)
    prediction = np.where(probability >= 0.70, 1, 0)

    # Visualisasi
    poly = gdf.iloc[0].geometry
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.set_title("Visualisasi Polygon & Titik QC", fontweight='bold')
    
    if hasattr(poly, "exterior"):
        x, y = poly.exterior.xy
        ax.plot(x, y, color="tab:blue", linewidth=2)
        ax.fill(x, y, alpha=0.15, facecolor="tab:blue")

    colors = ["black", "green", "purple"]
    for i in range(1, 4):
        lat, lon = gdf.iloc[0][f"Latitude_{i}"], gdf.iloc[0][f"Longitude_{i}"]
        ax.scatter(lon, lat, s=100, c=colors[i-1], edgecolors='white', zorder=3, label=f"QC-{i}")
    
    ax.legend()
    ax.set_aspect("equal")
    ax.grid(True, linestyle='--', alpha=0.6)
    plt.show()

    # Status kini dievaluasi berdasarkan nilai prediction yang sudah melewati threshold 0.80
    status = "✅ VALID" if int(prediction[0]) == 1 else "❌ TIDAK VALID"
    
    print(f"{'='*30}\nHASIL PREDIKSI: {status}\nProbabilitas Valid: {probability[0]:.3%}\nAmbang Batas Validasi: > 70.00%\n{'='*30}")
    display(gdf[X_feat.columns])
# ===========================================================================
# 3) INTERACTIVE UI (Jupyter Only)
# ===========================================================================
if WIDGETS_AVAILABLE:
    DEFAULTS = {
        "shape": "POLYGON ((105.279121611669 -4.61294944550871, 105.282742642802 -4.61295160307554, 105.28273552638 -4.61305213663778, 105.282628246063 -4.61321015621269, 105.282556732449 -4.613332257707, 105.282478051271 -4.61342563898184, 105.28245258899 -4.61352565822589, 105.282406671129 -4.61364817075038, 105.282397946226 -4.61384785040607, 105.282324510892 -4.61422994589588, 105.282202859793 -4.61424076542341, 105.282013192225 -4.61416903314357, 105.281791294663 -4.61407902912378, 105.281587341926 -4.61402558687888, 105.280223999308 -4.61380352262068, 105.280037894649 -4.61369229416355, 105.279969926283 -4.61373181519669, 105.279927042557 -4.61386826669822, 105.279873430047 -4.61402014241115, 105.279110981559 -4.61402007137196, 105.279121611669 -4.61294944550871))",
        "qc1": "-4.613253116607666,105.2796859741211", "qc2": "-4.613449573516846,105.2806396484375", "qc3": "-4.61384391784668,105.28195190429688",
        "akurasi": 1, "satelit": 39
    }

    # Widgets Definition
    shape_text = widgets.Textarea(value=DEFAULTS["shape"], description="SHAPE WKT:", layout={'width': '95%', 'height': '80px'})
    qc1_text = widgets.Text(value=DEFAULTS["qc1"], description="QC 1 (lat,lon):", layout={'width': '95%'})
    qc2_text = widgets.Text(value=DEFAULTS["qc2"], description="QC 2 (lat,lon):", layout={'width': '95%'})
    qc3_text = widgets.Text(value=DEFAULTS["qc3"], description="QC 3 (lat,lon):", layout={'width': '95%'})
    akurasi_input = widgets.FloatText(value=DEFAULTS["akurasi"], description="AKURASI:")
    satelit_input = widgets.IntText(value=DEFAULTS["satelit"], description="SATELIT:")
    btn_run = widgets.Button(description="Run Prediction", button_style="success", icon="check")
    output = widgets.Output()

    def on_run_clicked(_):
        with output:
            clear_output(wait=True)
            try:
                l1, n1 = parse_latlon(qc1_text.value)
                l2, n2 = parse_latlon(qc2_text.value)
                l3, n3 = parse_latlon(qc3_text.value)
                
                input_df = pd.DataFrame([{
                    "SHAPE": shape_text.value, "AKURASI": akurasi_input.value, "JUMLAH_SATELIT": satelit_input.value,
                    "Latitude_1": l1, "Longitude_1": n1, "Latitude_2": l2, "Longitude_2": n2, "Latitude_3": l3, "Longitude_3": n3
                }])
                predict_and_plot(input_df)
            except Exception as e:
                print(f"Error: {e}")

    btn_run.on_click(on_run_clicked)
    display(shape_text, qc1_text, qc2_text, qc3_text, widgets.HBox([akurasi_input, satelit_input]), btn_run, output)
else:
    print("Run in a Jupyter environment to see the interactive dashboard.")

Textarea(value='POLYGON ((105.279121611669 -4.61294944550871, 105.282742642802 -4.61295160307554, 105.28273552…

Text(value='-4.613253116607666,105.2796859741211', description='QC 1 (lat,lon):', layout=Layout(width='95%'))

Text(value='-4.613449573516846,105.2806396484375', description='QC 2 (lat,lon):', layout=Layout(width='95%'))

Text(value='-4.61384391784668,105.28195190429688', description='QC 3 (lat,lon):', layout=Layout(width='95%'))

Button(button_style='success', description='Run Prediction', icon='check', style=ButtonStyle())

Output()